# 05 · Audit, C0 decision, report
1. Leakage audit of the S1 training sources against every headline test split (flagged clusters removed for all systems).
2. Headline scoring of every system on audited test docs (refuses dirty or unknown code provenance).
3. The preregistered C0 decision (paired cluster bootstrap, Holm over the family).
4. Leak vs over-redaction curves per set.

In [ ]:
# Shared session setup (run in every notebook)
import os, subprocess, pathlib
from google.colab import drive, userdata
drive.mount('/content/drive')
DRIVE = pathlib.Path('/content/drive/MyDrive/s1pii'); DRIVE.mkdir(parents=True, exist_ok=True)
os.environ['S1PII_DATA'] = '/content/s1pii_data'        # local disk; synced to/from Drive
os.environ['S1PII_RESULTS'] = str(DRIVE / 'results')     # ledger + prediction cache on Drive
REPO = pathlib.Path('/content/s1-pii')

def git(*args):
    # The token (Colab secret GH_TOKEN: fine-grained, read-only on anitksahu/s1-pii) is passed
    # through GIT_ASKPASS, never on the command line, and errors are printed redacted.
    tok = userdata.get('GH_TOKEN')
    if not tok:
        raise RuntimeError('Add a GH_TOKEN secret in Colab (key icon, left sidebar) with read access to anitksahu/s1-pii')
    askpass = pathlib.Path('/tmp/askpass.sh'); askpass.write_text('#!/bin/sh\necho "$S1PII_GIT_TOKEN"\n'); askpass.chmod(0o700)
    env = {**os.environ, 'GIT_ASKPASS': str(askpass), 'S1PII_GIT_TOKEN': tok, 'GIT_TERMINAL_PROMPT': '0'}
    r = subprocess.run(['git', *args], env=env, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError('git ' + args[0] + ' failed: ' + (r.stderr or '').replace(tok, '***')[-800:])
    return r.stdout

if not REPO.exists():
    git('clone', 'https://x-access-token@github.com/anitksahu/s1-pii.git', str(REPO))
else:
    git('-C', str(REPO), 'pull', '--ff-only')
os.chdir(REPO)
subprocess.run(['rsync', '-a', f'{DRIVE}/data/', os.environ['S1PII_DATA'] + '/'], check=False)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)


In [ ]:
!pip install -q -e '.[data,model]'
# high-RAM runtime recommended: the audit indexes ~150k training docs
!python -m s1pii.c0 audit --models /content/drive/MyDrive/s1pii/models

In [ ]:
# C2 on the PII-TRACE calibration split, before any C0 scoring (prereg)
from s1pii import bench, c0
idx = c0.prediction_index()
c2_pred = idx[('s1_all-sources-s1', str(bench.split_paths('pii_trace')['calib'].resolve()))]
print(c0.decide_propagation(c2_pred))
# If propagation is False, re-run notebook 04 with --no-propagation for the headline S1 predictions.

In [ ]:
import json, pathlib, pandas as pd
from s1pii import bench, c0
idx = c0.prediction_index()
rows = []
for (system, docs_path), test_pred in idx.items():
    for ds in c0.HEADLINE:
        p = bench.split_paths(ds)
        if str(p['test'].resolve()) != docs_path:
            continue
        if (ds, system) in c0.EXCLUDED:
            continue
        calib = idx.get((system, str(p['calib'].resolve())))
        r = bench.score(system, ds, calib, test_pred, headline=True)['result']
        rows.append({'dataset': ds, 'system': system, 'pAUC': r['pauc'], 'ci': r['pauc_ci'], 'reach_over': r['reachable_max_over'],
                     'leak@1%': r['best_leak_at_budget']['0.01'], 'exposure@default': r['operating_points']['default']['span_exposure']})
TABLE = pd.DataFrame(rows).sort_values(['dataset', 'pAUC']); TABLE

In [ ]:
FRESH_REAL_ABSENT = not bench.split_paths('fresh_real')['test'].exists()   # declared fallback (prereg)
res = c0.decide(n_boot=10_000, fresh_real_absent=FRESH_REAL_ABSENT)
print(json.dumps(res['decision'], indent=1, default=str)[:3000]); print('problems:', res['problems'])
pathlib.Path(os.environ['S1PII_RESULTS'], 'c0.json').write_text(json.dumps(res, indent=1, default=str))

In [ ]:
import matplotlib.pyplot as plt
from s1pii.schema import read_jsonl
from s1pii.ledger import read_predictions
from s1pii.eval import metrics as M
from s1pii.eval.evaluate import views
for ds in [d for d in c0.HEADLINE if bench.split_paths(d)['test'].exists()]:
    docs = c0.audited_test_docs(ds); keep = {d.doc_id for d in docs}
    fig, ax = plt.subplots(figsize=(5, 4))
    for (system, docs_path), path in sorted(idx.items()):
        if docs_path != str(bench.split_paths(ds)['test'].resolve()):
            continue
        preds = {k: v for k, v in read_predictions(path)[1].items() if k in keep}
        _, hp, hn = M.cluster_histograms(views(docs, preds, ds))
        leak, over = M.curve_from_hist(hp.sum(0), hn.sum(0))
        ax.step(over, leak, where='post', label=system)
    ax.set_xlim(0, 0.05); ax.set_xlabel('over-redaction (non-PII alnum chars masked)'); ax.set_ylabel('leak (PII alnum chars unmasked)'); ax.set_title(ds); ax.legend(fontsize=7)
    fig.savefig(pathlib.Path(os.environ['S1PII_RESULTS']) / f'curve-{ds}.png', dpi=150, bbox_inches='tight'); plt.show()